# Avance 4 · Regresión de la potencia con huellas

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este cuaderno es el compañero del [04b](04b_Regresion_actividad.ipynb). Repite sus seis modelos de regresión con las **mismas filas, los mismos pliegues y las mismas semillas**, y cambia una sola cosa: qué información de la estructura reciben.

Los modelos del 03, del 04 y del 04b se entrenaron solo con descriptores de RDKit, que resumen la molécula entera (su tamaño, su polaridad, cuántos anillos tiene). La decisión [1.3](../docs/decisiones.md#d1-3) contemplaba también las huellas **Sort & Slice**, que dicen qué fragmentos concretos tiene la molécula y son la base para explicar una predicción a nivel de subestructura. Aquí medimos qué cambia al agregarlas. El 04b sigue siendo el cuaderno que se entrega; este lo acompaña.

## 1. Preparación

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por modelo: la columna de tiempos es comparable

import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"

from src.evaluation.regresion import concordancia_harrell, rmse_exactas
from src.features.huellas import AgregarHuellas, agrupar_iguales, identificadores_morgan
from src.features.seleccion import DepuracionDescriptores, SeleccionL1
from src.models.pisos import MediaPorDiana

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

### Las huellas y el código que las arma

Una huella **Sort & Slice** es una tabla de 0 y 1: una columna por subestructura (el entorno de radio 2 alrededor de un átomo, el mismo que ECFP4) y un 1 si la molécula la tiene. De todas las subestructuras que aparecen se conservan las **2,048 más frecuentes**. A diferencia de una huella plegada por hash, cada columna es **una** subestructura, y por eso una columna con peso alto se puede traducir a un fragmento concreto.

**El vocabulario se aprende en cada pliegue, con sus filas de entrenamiento.** Es lo que más importa de este cuaderno. Qué subestructuras son «las más frecuentes» depende de con qué filas se cuenten: si se cuentan con todas, la validación ya influyó en qué columnas existen, y eso es una fuga que no aparece en ninguna métrica y hace que las huellas parezcan mejores de lo que son. Por eso las moléculas se leen una sola vez (una columna por cada subestructura que aparece en alguna) y la elección de las 2,048 ocurre dentro de la canalización, en cada ajuste.

Dos diferencias con el `SortAndSlice` de `src/features/molecular.py`, que cuenta por molécula sobre una lista fija: aquí la frecuencia se cuenta por fila (en la fase 2 un compuesto aparece en 1.18 pares en promedio) y la elección se repite en cada ajuste. `tests/test_huellas.py` comprueba que, cuando cada molécula aparece una vez, el vocabulario y la matriz son idénticos a los de `SortAndSlice`.

**Lo que se ejecuta es el código de `src/`**, que importa la celda de arriba. Se muestra aquí para que se lea sin abrir otro archivo, y la prueba `tests/test_04c_codigo.py` comprueba que lo mostrado y lo importado sean idénticos.

```python
def identificadores_morgan(smiles, radio: int = 2) -> tuple[sparse.csr_matrix, np.ndarray]:
    """Matriz binaria dispersa (filas × subestructuras distintas) y los identificadores.

    Una columna por subestructura, sin plegar, ordenadas por identificador. Cada SMILES se lee
    una vez aunque aparezca en varias filas. Falla si alguno no se puede leer: quitarlo en
    silencio desalinearía la matriz con la de descriptores.
    """
    smiles = np.asarray(smiles, dtype=object)
    unicos, inverso = np.unique(smiles, return_inverse=True)
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radio)
    ids_por_molecula = []
    ilegibles = []
    for s in unicos:
        mol = a_mol(s)
        if mol is None:
            ilegibles.append(s)
            ids_por_molecula.append(np.empty(0, dtype=np.int64))
        else:
            ids_por_molecula.append(
                np.fromiter(gen.GetSparseCountFingerprint(mol).GetNonzeroElements().keys(),
                            dtype=np.int64))
    if ilegibles:
        raise ValueError(f"{len(ilegibles)} SMILES no se pueden leer (el primero: {ilegibles[0]!r})")
    identificadores = np.unique(np.concatenate(ids_por_molecula))
    columnas = np.concatenate([np.searchsorted(identificadores, ids) for ids in ids_por_molecula])
    filas = np.repeat(np.arange(len(unicos)), [len(ids) for ids in ids_por_molecula])
    por_molecula = sparse.csr_matrix(
        (np.ones(len(columnas), dtype=np.uint8), (filas, columnas)),
        shape=(len(unicos), len(identificadores)))
    return por_molecula[inverso], identificadores

def agrupar_iguales(huellas: sparse.csr_matrix, identificadores: np.ndarray
                    ) -> tuple[sparse.csr_matrix, np.ndarray, list[tuple[int, ...]]]:
    """Deja una sola columna por cada grupo de subestructuras que valen lo mismo en todas las filas.

    Dos subestructuras que siempre aparecen juntas (el `S` y el `O=S` de un sulfonilo, por ejemplo)
    son la misma variable para cualquier modelo, y si ambas entran el modelo reparte su peso entre
    ellas de forma arbitraria. No se mira la etiqueta, y lo que es idéntico en todas las filas lo es
    también en cualquier subconjunto, así que hacerlo una vez sobre todo el conjunto no filtra
    información de la validación. Se queda la primera columna de cada grupo y se devuelve, para cada
    columna conservada, la tupla con los identificadores de todas las que representa (la propia
    primero), para poder nombrar la variable con todas sus subestructuras.
    """
    columnas = huellas.tocsc()
    columnas.sort_indices()
    grupos: dict[bytes, list[int]] = {}
    for j in range(columnas.shape[1]):
        grupos.setdefault(columnas.indices[columnas.indptr[j]:columnas.indptr[j + 1]].tobytes(),
                          []).append(j)
    conservadas = [g[0] for g in grupos.values()]
    equivalentes = [tuple(int(i) for i in identificadores[g]) for g in grupos.values()]
    return columnas[:, conservadas].tocsr(), identificadores[conservadas], equivalentes

class AgregarHuellas(BaseEstimator, TransformerMixin):
    """Entrada mixta → matriz densa con las columnas `densas` y las `n_bits` huellas más frecuentes.

    `X` es una matriz dispersa cuyas primeras columnas son las densas (descriptores y diana) y el
    resto es la salida de `identificadores_morgan`. En `fit` cuenta cuántas filas de entrenamiento
    tiene cada subestructura y se queda con las `n_bits` más frecuentes (los empates se resuelven por
    la posición de la columna, que sigue el orden del identificador, para que sea reproducible).
    `transform` devuelve las columnas densas indicadas y esas huellas, en `float32`.

    Con `densas=()` solo salen las huellas.
    """

    def __init__(self, n_densas: int, densas: tuple[int, ...] | None = None, n_bits: int = 2048):
        self.n_densas = n_densas
        self.densas = densas
        self.n_bits = n_bits

    def fit(self, X, y=None):
        huellas = sparse.csr_matrix(X)[:, self.n_densas:]
        frecuencia = np.asarray(huellas.sum(axis=0)).ravel()
        orden = np.lexsort((np.arange(len(frecuencia)), -frecuencia.astype(np.int64)))
        usadas = orden[:self.n_bits]
        self.columnas_ = usadas[frecuencia[usadas] > 0]
        return self

    def transform(self, X):
        X = sparse.csr_matrix(X)
        densas = np.arange(self.n_densas) if self.densas is None else np.asarray(self.densas, int)
        partes = [X[:, densas].toarray().astype(np.float32),
                  X[:, self.n_densas:][:, self.columnas_].toarray().astype(np.float32)]
        return np.hstack(partes)
```

**El piso por diana.** El 04b solo tiene un piso, predecir siempre el promedio de todas las mediciones. Pero el corte de 10 µM no significa lo mismo en cada diana (deja a `LRRK2` en 98.1% de activos y a `NFE2L2` en 50.7%), y un modelo que solo sepa contra qué diana se midió ya supera ese promedio sin entender química. `MediaPorDiana` predice la potencia media de la diana de cada par con las filas de entrenamiento del pliegue, sin mirar los descriptores; lo que un modelo gane sobre ese piso es lo que le aporta la estructura de la molécula.

```python
class MediaPorDiana(BaseEstimator, RegressorMixin):
    """Predice la media de la variable objetivo de la diana de cada fila.

    `columnas` son las posiciones de las columnas de diana (una por diana, con 1 en la que
    corresponde y 0 en las demás). Una diana que no aparece en las filas de ajuste recibe la
    media de todas ellas.
    """

    def __init__(self, columnas: tuple[int, ...]):
        self.columnas = columnas

    def fit(self, X, y):
        diana = np.asarray(X)[:, list(self.columnas)]
        y = np.asarray(y, dtype=float)
        self.media_global_ = float(y.mean())
        n = diana.sum(axis=0)
        suma = diana.T @ y
        self.medias_ = np.where(n > 0, suma / np.maximum(n, 1), self.media_global_)
        return self

    def predict(self, X):
        diana = np.asarray(X)[:, list(self.columnas)]
        sin_diana = diana.sum(axis=1) == 0
        p = diana @ self.medias_
        p[sin_diana] = self.media_global_
        return p
```

La depuración de descriptores (brazo `depurados`, más abajo) usa dos filtros que ya existían, `casi_constantes` y `filtro_correlacion`, y la selección por coeficientes (brazo `seleccionados`) usa un Lasso. Las dos se ajustan también dentro de cada pliegue:

```python
def poder_spearman(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """|Spearman| de cada columna contra un objetivo continuo: la versión de `poder_univariado`
    para una regresión. Una columna constante da 0."""
    rx = np.apply_along_axis(rankdata, 0, X)
    ry = rankdata(y)
    rx = rx - rx.mean(axis=0)
    ry = ry - ry.mean()
    den = np.sqrt((rx ** 2).sum(axis=0) * (ry ** 2).sum())
    with np.errstate(invalid="ignore", divide="ignore"):
        r = (rx * ry[:, None]).sum(axis=0) / den
    return np.abs(np.nan_to_num(r))

class DepuracionDescriptores(BaseEstimator, TransformerMixin):
    """Los dos filtros de siempre, pero aprendidos con las filas de entrenamiento de cada pliegue.

    Actúa sobre las primeras `n_descriptores` columnas y deja pasar el resto (la diana en 0/1).
    Quita las casi constantes (un valor reúne `frecuencia_modal` de las filas) y, de cada par con
    |Spearman| mayor que `umbral`, la que menos se relaciona con el objetivo. Las `protegidas`
    (posiciones dentro de los descriptores) ganan cualquier par. Es el criterio de
    `casi_constantes` y `filtro_correlacion`; lo que cambia es dónde se ajusta.
    """

    def __init__(self, n_descriptores: int, protegidas: tuple[int, ...] = (),
                 frecuencia_modal: float = 0.99, umbral: float = 0.95):
        self.n_descriptores = n_descriptores
        self.protegidas = protegidas
        self.frecuencia_modal = frecuencia_modal
        self.umbral = umbral

    def fit(self, X, y):
        desc = np.asarray(X[:, :self.n_descriptores], dtype=float)
        variables = np.flatnonzero(casi_constantes(desc, self.frecuencia_modal))
        self.quitadas_constantes_ = np.setdiff1d(np.arange(self.n_descriptores), variables)
        sub = desc[:, variables]
        en_sub = tuple(int(np.flatnonzero(variables == p)[0]) for p in self.protegidas
                       if p in variables)
        sin_redundancia = filtro_correlacion(sub, poder_spearman(sub, np.asarray(y, float)),
                                             self.umbral, en_sub)
        self.descriptores_ = variables[sin_redundancia]
        self.quitadas_redundantes_ = variables[~sin_redundancia]
        self.columnas_ = np.r_[self.descriptores_, np.arange(self.n_descriptores, X.shape[1])]
        return self

    def transform(self, X):
        return X[:, self.columnas_]

class SeleccionL1(BaseEstimator, TransformerMixin):
    """Se queda con los descriptores a los que una penalización L1 no les lleva el peso a cero.

    Actúa sobre las primeras `n_descriptores` columnas y deja pasar el resto (la diana en 0/1).
    En cada ajuste escala los descriptores, entrena un Lasso contra el objetivo con las filas de
    entrenamiento y conserva los de coeficiente distinto de cero. Las `protegidas` se quedan
    siempre. `alpha` es un hiperparámetro que se fija de antemano; no se elige mirando la
    validación.
    """

    def __init__(self, n_descriptores: int, alpha: float = 0.01, protegidas: tuple[int, ...] = ()):
        self.n_descriptores = n_descriptores
        self.alpha = alpha
        self.protegidas = protegidas

    def fit(self, X, y):
        desc = np.asarray(X[:, :self.n_descriptores], dtype=float)
        lasso = Lasso(alpha=self.alpha, max_iter=5000).fit(
            StandardScaler().fit_transform(desc), np.asarray(y, float))
        elegidas = set(np.flatnonzero(lasso.coef_ != 0)) | set(self.protegidas)
        self.descriptores_ = np.array(sorted(elegidas), dtype=int)
        self.quitadas_ = np.setdiff1d(np.arange(self.n_descriptores), self.descriptores_)
        self.columnas_ = np.r_[self.descriptores_, np.arange(self.n_descriptores, X.shape[1])]
        return self

    def transform(self, X):
        return X[:, self.columnas_]
```

### Los datos

Son los mismos pares, con el mismo criterio, que el 04b: entrenamiento y validación (la prueba reservada no se toca), sin los pares que no informan nada, y con la diana como columnas de 0 y 1. Los modelos se entrenan con las mediciones exactas y la validación incluye las censuradas.

In [ ]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = pares[pares["particion"].ne("prueba")].merge(desc, on="smiles").reset_index(drop=True)
cols = [c for c in desc.columns if c != "smiles"]

inferior = d["limite_inferior"].to_numpy(float)
superior = d["limite_superior"].to_numpy(float)
util = ~(np.isneginf(inferior) & np.isposinf(superior)) & (inferior <= superior)
print(f"{len(d):,} pares de entrenamiento y validación · {(~util).sum():,} sin información y salen")

d = d[util].reset_index(drop=True)
inferior, superior = inferior[util], superior[util]
# La diana entra como columnas 0/1: sin ella el modelo no sabe contra qué se midió.
X = np.hstack([d[cols].to_numpy("float32"), pd.get_dummies(d["target_gene"]).to_numpy("float32")])
exacta = inferior == superior
y_cls = d["activo_10um"].to_numpy(float)       # la clase derivada a 10 µM; NaN si no decide
g = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()          # solo_chembl, solo_npass, mixto
print(f"{len(d):,} pares · {exacta.sum():,} con potencia exacta ({exacta.mean():.1%}) · "
      f"{(~exacta).sum():,} solo con cotas")

### Las huellas de los pares

Se leen los SMILES una sola vez y se arma una matriz dispersa con una columna por subestructura. **Todavía no se elige ningún vocabulario**: eso pasa dentro de cada pliegue. Antes se agrupan las subestructuras que valen lo mismo en todos los pares (porque siempre aparecen juntas, como el `S` y el `O=S` de un sulfonilo): se queda una sola columna y se guarda, en `equivalentes`, el nombre de todas las que representa. Eso no mira la potencia, y lo que es idéntico en todas las filas lo es en cualquier pliegue, así que no hay fuga. Así las 2,048 columnas no se gastan en la misma variable repetida y una atribución no reparte su peso entre copias. Las columnas de descriptores y de diana van primero y las huellas después, en una sola matriz dispersa.

In [ ]:
t0 = time.perf_counter()
huellas, identificadores = identificadores_morgan(d["smiles"])
n_distintas = huellas.shape[1]
huellas, identificadores, equivalentes = agrupar_iguales(huellas, identificadores)
n_agrupadas = sum(len(e) > 1 for e in equivalentes)
N_DESC = len(cols)                       # descriptores
N_DENSAS = X.shape[1]                    # descriptores + diana
DIANA = tuple(range(N_DESC, N_DENSAS))
X_mixta = sparse.hstack([sparse.csr_matrix(X), huellas]).tocsr()
print(f"{n_distintas:,} subestructuras distintas en {huellas.shape[0]:,} pares; "
      f"{n_distintas - huellas.shape[1]:,} valían lo mismo que otra y quedan {huellas.shape[1]:,} columnas "
      f"({n_agrupadas} de ellas representan a más de una) · {time.perf_counter() - t0:.1f} s")

## 2. Cinco brazos con los mismos seis modelos

| Brazo | Qué recibe el modelo | Para qué está |
|:--|:--|:--|
| `descriptores` | 201 descriptores y la diana | Es el 04b. Es el **control**: tiene que repetir sus cifras |
| `depurados` | Los descriptores sin las columnas casi constantes ni las redundantes (|Spearman| mayor que 0.95), y la diana | Responde si 201 descriptores son demasiados. TPSA y los donadores de hidrógeno se protegen, porque son las dos propiedades con las que se comprueba que una explicación tenga sentido |
| `seleccionados` | Los descriptores a los que un Lasso (penalización L1, `alpha = 0.01`) no les lleva el peso a cero, y la diana | La otra forma de preguntar si sobran columnas: que sea un modelo el que elija, con sus coeficientes. Se ajusta dentro de cada pliegue, y no con los resultados del Avance 4, porque elegir variables mirando una validación y volver a evaluar en ella da cifras mejores de lo que son. `alpha` se fijó de antemano (conserva unos 90 de los 201 en un pliegue de entrenamiento, sin mirar ninguna validación) |
| `descriptores+huellas` | Los 201 descriptores, las 2,048 huellas y la diana | Lo que se quiere medir: qué agregan los fragmentos a lo que ya había |
| `huellas` | Solo las 2,048 huellas y la diana | Cuánto dice la estructura por sí sola, sin los descriptores |

Los modelos y sus hiperparámetros son los del 04b; la explicación de lo que hace cada uno está allí. Los que dependen de distancias o de magnitudes llevan el escalado dentro de la canalización, así que cada pliegue aprende el suyo. En las columnas de huellas, que ya valen 0 o 1, el escalado mínimo-máximo no cambia nada.

Además de los cinco brazos, se miden dos pisos que no ven la química, **una sola vez, con los mismos pliegues**: `media` (el promedio de todo, el piso del 04b) y `media_por_diana` (el promedio de cada diana). Las dos tablas de resultados los muestran aparte, y una tabla más resta el segundo a cada modelo para ver cuánto de la concordancia queda una vez que se sabe la diana.

**Lo que se fija antes de ver los resultados.**

- **La métrica principal es la concordancia de Harrell**, igual que en el 04b, y se compara **pliegue por pliegue contra el brazo `descriptores`**: las diferencias se calculan sobre las mismas 15 evaluaciones (3 semillas × 5 pliegues), no entre promedios.
- **Un brazo mejora a un modelo solo si** la diferencia media de concordancia supera **0.006** (la mayor dispersión entre evaluaciones que se vio en el 04b) **y** es positiva en al menos **12 de las 15** evaluaciones. Si no, se reporta como «sin mejora medible». Si las huellas no mejoran la métrica, el argumento a favor de usarlas sería la explicabilidad, y así se diría; no se presentaría como una mejora.
- **Contra el sobreajuste**, además de los pliegues agrupados por esqueleto (un modelo que memoriza fragmentos de un esqueleto no gana nada en el pliegue que lo deja fuera), se mide la **brecha entre entrenamiento y validación** de cada modelo, y se incluyen los brazos `depurados` y `seleccionados`.

Un aviso sobre los tiempos: el kernel del `SVR` y las distancias del `kNN` crecen con el número de columnas, así que los brazos con huellas son los más lentos de este cuaderno.

In [ ]:
MODELOS = {
    "ridge":          make_pipeline(MinMaxScaler(), Ridge()),
    "arbol":          DecisionTreeRegressor(min_samples_leaf=5),
    "knn":            make_pipeline(MinMaxScaler(), KNeighborsRegressor(n_neighbors=15)),
    "bayesian_ridge": make_pipeline(StandardScaler(), BayesianRidge()),
    "svr":            make_pipeline(MinMaxScaler(), SVR()),
    "mlp":            make_pipeline(MinMaxScaler(), MLPRegressor(hidden_layer_sizes=(64,), max_iter=600)),
}
REFERENCIA = {"media": DummyRegressor(),                 # el piso del 04b: el promedio de todo
              "media_por_diana": MediaPorDiana(DIANA)}  # el piso que sabe la diana y nada de química
SEMILLAS = (0, 1, 2)
CORTE = 5.0                                      # pActividad 5 = 10 µM, el mismo corte del 04
N_BITS = 2048                                    # tamaño del vocabulario de Sort & Slice
ALFA_L1 = 0.01                                   # penalización del Lasso; fija, no se elige con la validación
PROTEGIDAS = tuple(cols.index(c) for c in ("TPSA", "NumHDonors"))   # posiciones dentro de los descriptores
UMBRAL_MEJORA = 0.006                            # diferencia de concordancia que hay que superar
MINIMO_PLIEGUES = 12                             # de 15 evaluaciones, en cuántas tiene que mejorar
MUESTRA_ENTRENAMIENTO = 5000                     # filas de entrenamiento para medir la brecha

# Cada brazo: la matriz que recibe y cómo se arma su canalización a partir del modelo del 04b.
BRAZOS = {
    "descriptores":         (X,       lambda m: m),
    "depurados":            (X,       lambda m: make_pipeline(DepuracionDescriptores(N_DESC, PROTEGIDAS), m)),
    "seleccionados":        (X,       lambda m: make_pipeline(SeleccionL1(N_DESC, ALFA_L1, PROTEGIDAS), m)),
    "descriptores+huellas": (X_mixta, lambda m: make_pipeline(AgregarHuellas(N_DENSAS, n_bits=N_BITS), m)),
    "huellas":              (X_mixta, lambda m: make_pipeline(AgregarHuellas(N_DENSAS, DIANA, N_BITS), m)),
}

In [ ]:
def medir(inf, sup, clase, pred):
    """Concordancia con todas las mediciones; Spearman y RMSE con las exactas; MCC y AUROC de la clase derivada."""
    exa = inf == sup
    pe = pred[exa]
    rho = np.nan if np.ptp(pe) == 0 else spearmanr(inf[exa], pe).statistic   # nan si predice una constante
    rmse, _ = rmse_exactas(inf, sup, pred)
    fila = {"concordancia": concordancia_harrell(inf, sup, pred), "spearman": rho, "rmse": rmse,
            "mcc": np.nan, "auroc": np.nan}
    decide = ~np.isnan(clase)
    if len(np.unique(clase[decide])) == 2:
        fila["mcc"] = matthews_corrcoef(clase[decide], pred[decide] >= CORTE)
        fila["auroc"] = roc_auc_score(clase[decide], pred[decide])
    return fila


filas = []
elegidas = {}                                    # qué descriptores dejó cada brazo de selección en cada ajuste
quitadas = {}                                    # y cuáles quitó cada paso, para contar cómo se fueron quitando
estratificar = np.where(np.isnan(y_cls), -1, y_cls).astype(int)
inicio = time.perf_counter()
for semilla in SEMILLAS:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=semilla)         # los mismos pliegues del 04b
    for k, (ajuste, val) in enumerate(cv.split(X, estratificar, g), 1):
        ajuste_ex = ajuste[exacta[ajuste]]                                    # se entrena solo con las exactas
        # Un subconjunto del entrenamiento para medir la brecha; el mismo en todos los brazos.
        muestra = np.random.default_rng(semilla * 10 + k).choice(
            ajuste_ex, min(MUESTRA_ENTRENAMIENTO, len(ajuste_ex)), replace=False)
        for brazo, (matriz, armar) in BRAZOS.items():
            X_aj, X_val, X_mue = matriz[ajuste_ex], matriz[val], matriz[muestra]
            # Los pisos no ven la química, así que se miden una vez, en el brazo de control.
            for nombre, modelo in {**MODELOS, **(REFERENCIA if brazo == "descriptores" else {})}.items():
                m = armar(clone(modelo))
                # La semilla fija también el azar del modelo (árbol, MLP), no solo el reparto.
                m.set_params(**{p: semilla for p in m.get_params() if p.endswith("random_state")})
                reloj = time.perf_counter()
                m.fit(X_aj, inferior[ajuste_ex])
                seg = time.perf_counter() - reloj
                pred = m.predict(X_val)
                pred_mue = m.predict(X_mue)
                paso = m.steps[0][1] if hasattr(m, "steps") else m
                if hasattr(paso, "descriptores_") and nombre == "ridge":      # igual para los seis: no depende del modelo
                    elegidas[(brazo, semilla, k)] = [cols[i] for i in paso.descriptores_]
                    quitadas[(brazo, semilla, k)] = {
                        paso_: [cols[i] for i in getattr(paso, atributo, [])]
                        for paso_, atributo in (("casi constantes", "quitadas_constantes_"),
                                                ("redundantes", "quitadas_redundantes_"),
                                                ("Lasso en cero", "quitadas_"))}
                base = {"brazo": brazo, "modelo": nombre, "semilla": semilla, "pliegue": k, "seg": seg}
                ent = {"ent_concordancia": concordancia_harrell(inferior[muestra], superior[muestra], pred_mue),
                       "ent_rmse": rmse_exactas(inferior[muestra], superior[muestra], pred_mue)[0]}
                filas.append({**base, "estrato": "todos", **ent,
                              **medir(inferior[val], superior[val], y_cls[val], pred)})
                for e in np.unique(procedencia[val]):
                    sel = procedencia[val] == e
                    ys = y_cls[val][sel]
                    ys = ys[~np.isnan(ys)]
                    if min(ys.sum(), (1 - ys).sum()) >= 20:         # estrato con 20 casos de cada clase
                        filas.append({**base, "estrato": e,
                                      **medir(inferior[val][sel], superior[val][sel], y_cls[val][sel], pred[sel])})
            print(f"semilla {semilla} · pliegue {k} · {brazo:<22} {(time.perf_counter() - inicio) / 60:6.1f} min", flush=True)
res = pd.DataFrame(filas)
print(f"{len(res):,} filas: {res['brazo'].nunique()} brazos × {res['modelo'].nunique()} modelos × "
      f"{len(SEMILLAS)} semillas × 5 pliegues, por estrato")

## 3. Resultados

### Cómo se fueron quitando variables

Los modelos del 04 y del 04b se entrenaron con **todos** los descriptores que dejó la limpieza, sin quitar ninguno más. Esta tabla cuenta cada paso por el que pasa una columna, desde las que calcula RDKit hasta las que ve un modelo, y quién la quitó:

- **Limpieza.** Se hace una sola vez, sin mirar la potencia: quita las columnas constantes, las que tienen valores faltantes o infinitos y las que se salen del rango de `float32`. Es de donde salen los 201 descriptores que usan el 04b y el brazo `descriptores`.
- **`depurados`.** Se ajusta en cada pliegue, en dos pasos: primero las casi constantes (un solo valor reúne el 99% de las filas) y luego las redundantes (de cada par con |Spearman| mayor que 0.95 se queda la que más se relaciona con la potencia).
- **`seleccionados`.** Se ajusta en cada pliegue: se queda con lo que un Lasso no manda a cero.

En los dos casos, TPSA y NumHDonors no se pueden quitar. Como cada pliegue elige con sus propias filas, las cuentas son un promedio con el mínimo y el máximo entre las 15 evaluaciones, y después se ve cuáles descriptores sobreviven siempre.

In [ ]:
from src.features.molecular import limpiar_descriptores

crudos = pd.read_parquet(PROCESADOS / "fase2_descriptores.parquet").drop(columns="smiles")
limpio, motivos = limpiar_descriptores(crudos, verbose=False)
assert list(limpio.columns) == cols, "las columnas de la matriz no son las que da la limpieza"


def cuantas(brazo, paso):
    v = [len(q[paso]) for (b, _, _), q in quitadas.items() if b == brazo]
    return f"{np.mean(v):.0f} ({min(v)}–{max(v)})", np.mean(v)


q_const, m_const = cuantas("depurados", "casi constantes")
q_red, m_red = cuantas("depurados", "redundantes")
q_l1, m_l1 = cuantas("seleccionados", "Lasso en cero")
por_motivo = ", ".join(f"{k.replace('_', ' ')} {len(v)}" for k, v in motivos.items() if v)
embudo = pd.DataFrame([
    ("RDKit calcula", "", "", len(crudos.columns)),
    ("Limpieza, una vez", por_motivo, f"{len(crudos.columns) - N_DESC}", N_DESC),
    ("`depurados`: casi constantes", "un valor reúne el 99% de las filas", q_const, f"{N_DESC - m_const:.0f}"),
    ("`depurados`: redundantes", "|Spearman| > 0.95 con otra", q_red, f"{N_DESC - m_const - m_red:.0f}"),
    ("`seleccionados`: Lasso", f"coeficiente cero con alpha = {ALFA_L1}", q_l1, f"{N_DESC - m_l1:.0f}"),
], columns=["paso", "criterio", "quita (media y rango)", "quedan"]).set_index("paso")
embudo

Y los descriptores que sobreviven en todas las evaluaciones, que son los que más se pueden defender como parte de la explicación:

In [ ]:
resumen = []
for brazo in ("depurados", "seleccionados"):
    listas = [v for (b, _, _), v in elegidas.items() if b == brazo]
    veces = pd.Series([c for v in listas for c in v]).value_counts()
    resumen.append({"brazo": brazo, "evaluaciones": len(listas),
                    "descriptores_que_deja": f"{min(map(len, listas))}–{max(map(len, listas))} de {N_DESC}",
                    "en_todas": int((veces == len(listas)).sum())})
display(pd.DataFrame(resumen).set_index("brazo"))
en_todas_l1 = [c for c, n in pd.Series([c for (b, _, _), v in elegidas.items() if b == "seleccionados" for c in v]).value_counts().items()
               if n == sum(1 for (b, _, _) in elegidas if b == "seleccionados")]
print(f"Descriptores que el Lasso conserva en todas las evaluaciones ({len(en_todas_l1)}):")
print(", ".join(en_todas_l1))

### El control: el brazo `descriptores` repite el 04b

Antes de leer nada, el brazo que no cambia nada tiene que dar lo que ya se registró. Si no coincide, algo en la preparación o en el entorno es distinto y los demás brazos no se pueden comparar con el 04b.

In [ ]:
REGISTRO_04B = RAIZ / "experimentos" / "2026-10-25-artemio-2.9-comparativa-regresion-seis-modelos" / "metricas.json"
registrado = json.loads(REGISTRO_04B.read_text(encoding="utf-8"))["todos"]
aqui = res[(res["brazo"] == "descriptores") & (res["estrato"] == "todos")].groupby("modelo")["concordancia"].mean().loc[list(MODELOS)]
control = pd.DataFrame({"04b (registrado)": {m: registrado[m]["concordancia"] for m in aqui.index},
                        "04c (esta corrida)": aqui}).round(3)
control["diferencia"] = (control["04c (esta corrida)"] - control["04b (registrado)"]).round(3)
control

### Todos los brazos, por modelo

La concordancia media de cada modelo en cada brazo (la métrica principal), y en la segunda tabla cuánto tarda un ajuste.

In [ ]:
def tabla(r, estrato="todos"):
    """Media por brazo y modelo. `desv_concordancia` es dispersión entre evaluaciones, no error estándar:
    las semillas comparten filas."""
    r = r[r["estrato"] == estrato]
    t = r.groupby(["brazo", "modelo"])[["concordancia", "spearman", "rmse", "mcc", "auroc", "seg"]].mean()
    t.insert(1, "desv_concordancia", r.groupby(["brazo", "modelo"])["concordancia"].std())
    return t.round(3)


tabla_todos = tabla(res)
orden_brazos = list(BRAZOS)
concordancia = tabla_todos["concordancia"].unstack("brazo")[orden_brazos]
concordancia.loc[["knn", "svr", "mlp", "ridge", "bayesian_ridge", "arbol"]]

In [ ]:
tabla_todos["seg"].unstack("brazo")[orden_brazos].loc[["knn", "svr", "mlp", "ridge", "bayesian_ridge", "arbol"]].round(1)

### Los pisos, y cuánto queda una vez que se sabe la diana

La primera tabla son los dos pisos. La segunda resta la concordancia de `media_por_diana` a la de cada modelo y brazo, en el agregado y por procedencia: es la parte del resultado que no se explica por saber contra qué diana se midió.

In [ ]:
pisos = tabla_todos.xs("descriptores", level="brazo").loc[list(REFERENCIA), ["concordancia", "spearman", "rmse", "mcc", "auroc"]]
display(pisos)

ganancia = {}
for e in ["todos"] + sorted(set(res["estrato"]) - {"todos"}):
    t_e = tabla(res, e)["concordancia"].unstack("brazo")[orden_brazos].loc[list(MODELOS)]
    piso_e = tabla(res, e).xs("descriptores", level="brazo").loc["media_por_diana", "concordancia"]
    ganancia[e] = (t_e - piso_e).round(3)
pd.concat(ganancia, names=["procedencia"])

### ¿Mejoran las huellas? La comparación pliegue por pliegue

Para cada modelo y cada brazo se resta la concordancia del brazo `descriptores` en **la misma evaluación** (misma semilla, mismo pliegue). `mejor_en` cuenta en cuántas de las 15 evaluaciones el brazo gana, y `mejora` aplica la regla fijada arriba: diferencia media mayor que 0.006 **y** al menos 12 de 15.

In [ ]:
def comparacion(r, estrato="todos"):
    """Diferencia de concordancia de cada brazo contra `descriptores`, evaluación por evaluación."""
    r = r[(r["estrato"] == estrato) & ~r["modelo"].isin(list(REFERENCIA))]
    p = r.pivot_table(index=["modelo", "semilla", "pliegue"], columns="brazo", values="concordancia")
    filas = []
    for brazo in orden_brazos[1:]:
        dif = (p[brazo] - p["descriptores"]).dropna()
        for modelo, v in dif.groupby("modelo"):
            filas.append({"brazo": brazo, "modelo": modelo, "evaluaciones": len(v),
                          "dif_media": v.mean(), "dif_desv": v.std(), "mejor_en": int((v > 0).sum())})
    t = pd.DataFrame(filas).set_index(["brazo", "modelo"])
    t["mejora"] = (t["dif_media"] > UMBRAL_MEJORA) & (t["mejor_en"] >= MINIMO_PLIEGUES)
    return t.round(3)


comparacion(res)

### Por procedencia

Un promedio puede esconder cómo le va a los productos naturales. En el 04b los seis modelos quedaron cerca del azar en los pares que solo tiene NPASS, y es ahí donde un fragmento común con compuestos de entrenamiento podría ayudar donde los descriptores no. También es donde más cuidado hay que tener: los fragmentos típicos de los naturales podrían delatar de qué fuente viene el compuesto, que es la fuente con menos potencia. Por eso se mira cada procedencia por separado.

In [ ]:
por_procedencia = {e: comparacion(res, e) for e in ("solo_chembl", "mixto", "solo_npass")
                   if (res["estrato"] == e).any()}
conc_proc = pd.concat({e: tabla(res, e)["concordancia"].unstack("brazo")[orden_brazos]
                       for e in por_procedencia}, names=["procedencia"])
conc_proc

In [ ]:
pd.concat(por_procedencia, names=["procedencia"])[["dif_media", "dif_desv", "mejor_en", "mejora"]]

### La brecha entre entrenamiento y validación

Un modelo que se aprende los ejemplos acierta mucho en lo que vio y mucho menos en lo que no. La brecha es la concordancia (o el RMSE) sobre una muestra de 5,000 filas de entrenamiento menos la de validación. Con más columnas la brecha crece si el modelo está sobreajustando; que crezca no es un defecto si la validación también mejora. Con el `kNN` la muestra de entrenamiento incluye a cada punto como su propio vecino, así que su brecha sale inflada en todos los brazos y solo se compara dentro del mismo modelo.

In [ ]:
t = res[(res["estrato"] == "todos") & ~res["modelo"].isin(list(REFERENCIA))]
brecha = t.groupby(["brazo", "modelo"]).agg(
    conc_entrenamiento=("ent_concordancia", "mean"), conc_validacion=("concordancia", "mean"),
    rmse_entrenamiento=("ent_rmse", "mean"), rmse_validacion=("rmse", "mean"))
brecha["brecha_conc"] = brecha["conc_entrenamiento"] - brecha["conc_validacion"]
brecha["brecha_rmse"] = brecha["rmse_validacion"] - brecha["rmse_entrenamiento"]
brecha[["conc_entrenamiento", "conc_validacion", "brecha_conc", "brecha_rmse"]].unstack("brazo").round(3)

In [ ]:
# Registro en experimentos/ (decisión 1.3: qué representación reciben los modelos).
from src.tracking import Corrida

with Corrida("comparativa-regresion-huellas", decision="1.3",
             params={"modelos": list(MODELOS), "referencia": list(REFERENCIA), "brazos": list(BRAZOS),
                     "pliegues": 5, "entrena_con": "solo exactas", "objetivo": "pActividad",
                     "metrica_principal": "concordancia", "corte_mcc": CORTE, "n_bits_sortslice": N_BITS,
                     "radio_morgan": 2, "vocabulario": "se aprende en cada ajuste",
                     "depuracion": {"frecuencia_modal": 0.99, "umbral_spearman": 0.95,
                                    "protegidas": ["TPSA", "NumHDonors"]},
                     "seleccion_l1": {"alpha": ALFA_L1, "protegidas": ["TPSA", "NumHDonors"]},
                     "umbral_mejora": UMBRAL_MEJORA, "minimo_pliegues": MINIMO_PLIEGUES,
                     "estratos": "procedencia"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    c.metricas({"todos": {f"{b}/{m}": v for (b, m), v in tabla_todos.to_dict(orient="index").items()},
                "comparacion_todos": {f"{b}/{m}": v for (b, m), v in
                                      comparacion(res).to_dict(orient="index").items()},
                "brecha": {f"{b}/{m}": v for (b, m), v in brecha.to_dict(orient="index").items()},
                **{e: {f"{b}/{m}": v for (b, m), v in tabla(res, e).to_dict(orient="index").items()}
                   for e in por_procedencia}})

print(c.carpeta.name)

## 3.1 Lectura

*Se escribe después de la ejecución, con lo que digan las tablas de arriba. Lo que corresponde responder:*

1. ¿El brazo `descriptores` repite el 04b? (la tabla del control)
2. ¿Las huellas mejoran la concordancia de algún modelo según la regla fijada, en todos los pares y por procedencia?
3. ¿Cuánto de la concordancia queda sobre el piso `media_por_diana`, en el agregado y por procedencia?
4. ¿Mejoran solo las huellas o solo la combinación? ¿Qué dice el brazo `huellas` sobre cuánta información lleva la estructura por sí sola?
5. ¿Qué pasa con la brecha de entrenamiento y validación? ¿Los brazos `depurados` y `seleccionados` cambian algo respecto a los 201 descriptores? ¿Cuántos descriptores quedan y cuáles se repiten?
6. ¿Cuánto cuestan en tiempo?

# Declaración de uso de IA y contribución

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Nos apoyó con la generación de código de las huellas y de la depuración dentro del pliegue, y con el diseño de la comparación | Las secciones que nos ayudó a implementar fueron celdas de código | La decisión de medir las huellas, la regla de mejora, la revisión de los resultados y los comentarios generales fueron del equipo |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | *Por completar* |
| Artemio Santiago Padilla Robles | *Por completar* |
| José Antonio Vázquez Martínez | *Por completar* |